# Estimate scoring and conceding probabilities (Impect)

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import warnings
from pathlib import Path
import pandas as pd
import tqdm
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

PROJECT_ROOT = Path('..').resolve()
CONFIG_CANDIDATES = [
    PROJECT_ROOT / 'private/impect-pipeline/config/my_iteration.json',
    PROJECT_ROOT / 'docs/documentation/data/impect_open_data.example.json',
]
CONFIG_PATH = next((p for p in CONFIG_CANDIDATES if p.is_file()), CONFIG_CANDIDATES[-1])
with open(CONFIG_PATH) as f:
    CFG = json.load(f)
OUT = PROJECT_ROOT / 'data/impect' / CFG['output_basename']
spadl_h5 = OUT / 'spadl-impect.h5'
features_h5 = OUT / 'features.h5'
labels_h5 = OUT / 'labels.h5'
predictions_h5 = OUT / 'predictions.h5'
print('config:', CONFIG_PATH)
print('output:', OUT)

def games_with_actions(path):
    with pd.HDFStore(path) as store:
        games = store['games']
        ids = {int(k.rsplit('_', 1)[-1]) for k in store.keys() if k.startswith('/actions/game_')}
    return games[games.game_id.isin(ids)].sort_values('game_date').reset_index(drop=True)

import socceraction.vaep.features as fs

In [ ]:
games = games_with_actions(spadl_h5)
split = max(1, int(len(games) * 0.8))
train_games = games.iloc[:split]
test_games = games.iloc[split:] if split < len(games) else games.iloc[:1]
print(len(train_games), 'train,', len(test_games), 'test')


In [ ]:
xfns = [
    fs.actiontype, fs.actiontype_onehot, fs.bodypart_onehot,
    fs.result, fs.result_onehot, fs.goalscore,
    fs.startlocation, fs.endlocation, fs.movement, fs.space_delta,
    fs.startpolar, fs.endpolar, fs.team, fs.time_delta,
]
Xcols = fs.feature_column_names(xfns, 1)
Ycols = ['scores', 'concedes']

def getXY(gf):
    X, Y = [], []
    for gid in tqdm.tqdm(gf.game_id, desc='load'):
        X.append(pd.read_hdf(features_h5, f'game_{gid}')[Xcols])
        Y.append(pd.read_hdf(labels_h5, f'game_{gid}')[Ycols])
    return pd.concat(X).reset_index(drop=True), pd.concat(Y).reset_index(drop=True)

trainX, trainY = getXY(train_games)
testX, testY = getXY(test_games)


## Train

In [ ]:
import xgboost
models = {}
for col in Ycols:
    m = xgboost.XGBClassifier(n_estimators=50, max_depth=3, n_jobs=-1, verbosity=0, enable_categorical=True)
    m.fit(trainX, trainY[col])
    models[col] = m


## Evaluate (test split)

In [ ]:
from sklearn.metrics import brier_score_loss, roc_auc_score
Y_hat = pd.DataFrame({col: models[col].predict_proba(testX)[:, 1] for col in Ycols})
for col in Ycols:
    print('###', col, '###')
    y, p = testY[col], Y_hat[col]
    print('  Brier:', brier_score_loss(y, p))
    print('  ROC:', roc_auc_score(y, p))


## Save predictions

In [ ]:
with pd.HDFStore(predictions_h5, 'w') as store:
    for gid in tqdm.tqdm(games.game_id, desc='predict'):
        Xi = pd.read_hdf(features_h5, f'game_{gid}')[Xcols]
        Yi = pd.DataFrame({col: models[col].predict_proba(Xi)[:, 1] for col in Ycols})
        store.put(f'game_{gid}', Yi, format='table')
